# Tristan Zhang's AI Generated Images vs. Real Images Classification
In this notebook, we use [Tristan Zhang's AI generated images vs. real images kaggle dataset](https://www.kaggle.com/datasets/tristanzhang32/ai-generated-images-vs-real-images). We first convert the dataset into lower-resolution and lower-dimension images with the same context using bicubic convolutional interpolation for faster I/O. Then, we use `lightning.pytorch.LightningDataModule` and `lightning.pytorch.LightningModule` to create the dataset and model objects, respectively. Later, we search data loader pipeline settings and hyperparameters space and find a good combination of data loader pipeline settings and hyperparameter values for our problem. Finally, we train a new
model using optimal data loader pipeline settings and hyperparameters values using 2-stage fine-tuning of ResNet-50 pretrained on ImageNet, print metrics, plot visualizations and predict sample images from the dataset.

## Table of Contents

1. [Imports and Global Package Settings](#1-imports-and-global-package-settings)
1. [Lower the Dimensions and Size of the Dataset for Faster I/O and Get to Know the Data](#2-lower-the-dimensions-and-size-of-the-dataset-for-faster-io-and-get-to-know-the-data)
1. [Train and Validation Transformations Required by ResNet-50 and Suitable For the Task at Hand](#3-train-and-validation-transformations-required-by-resnet-50-and-suitable-for-the-task-at-hand)
1. [Fast Dev Run of Model to Verify Everything is Working](#4-fast-dev-run-of-model-to-verify-everything-is-working)
1. [Data loader pipeline settings](#5-data-loader-pipeline-settings)
1. [Hyperparameters Research](#6-hyperparameters-research)
1. [Final Training Phase - Using Optimal Data Loader Pipeline Settings and Hyperparameters Values for Training](#6-final-training-phase---using-optimal-data-loader-pipeline-settings-and-hyperparameters-values-for-training)
1. [Final Metrics and Sample Predictions](#7-final-metrics-and-sample-predictions)
1. [Conclusion](#8-conclusion)

## 1. Imports and Global Package Settings

In [ ]:
import torch
from torchvision import transforms
from sklearn.metrics import ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt
import warnings
from PIL import ImageFile, Image
from dataset import *
from utils import *
from model import *

# Don't show warnings.
warnings.filterwarnings('ignore')

# PIL opens images even with some pixel values being lost.
ImageFile.LOAD_TRUNCATED_IMAGES = True

# No constraint on number of image pixels by PIL.
Image.MAX_IMAGE_PIXELS = None

# PyTorch matrix multiplication precision setting for both accuracy and efficiency
torch.set_float32_matmul_precision('high')

# PyTorch device management.
device = 'cuda' if torch.cuda.is_available() else 'cpu'

## 2. Lower the Dimensions and Size of the Dataset for Faster I/O and Get to Know the Data

In [ ]:
# Lower the dimensions and size of the dataset for faster I/O.

lower_img_size(root='data', new_root='data_resized', format="JPEG", quality=85, optimize=True)

In [ ]:
# Test and visualize data module object

test_data_module(
    data_root='data',
    train_batch_size=16,
    val_batch_size=16,
    train_dir='train',
    val_dir='test', # In this project, we use test directory as validation set. We don't use test set in this project.
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2
)

## 3. Train and Validation Transformations Required by ResNet-50 and Suitable For the Task at Hand 

In [ ]:
# IMAGENET dataset mean and std
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# Train set transform, containing data augmentations, tensorization and normalization.
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])


# Validation set transform containing tensorization and normalization. Images are resized to 224 * 224.
val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])

## 4. Fast Dev Run of Model to Verify Everything is Working

In [ ]:
# Running the model for one step to check everything works smoothly

_, _, _ = instantiate_and_train_dataset_and_model(
    data_root='data_resized',
    train_batch_size=32,
    val_batch_size=32,
    learning_rate_stage1=1e-4,
    learning_rate_stage2=1e-5,
    num_epochs=10,
    train_transform=train_transform,
    val_transform=val_transform,
    devices=1,
    num_workers=4,
    pin_memory=True,
    prefetch_factor=2,
    precision='32-true',
    weight_decay_rate_stage1=1e-3,
    weight_decay_rate_stage2=1e-3,
    logger=False,
    enable_model_summary=False,
    enable_progress_bar=True,
    accumulate_grad_batches=4,
    train_dir='train', 
    val_dir='test',
    num_classes=2,
    weight_path='resnet50_weights.pth',
    fast_dev_run=True
)

## 5. Data loader pipeline settings

In [ ]:
dataloader_params = study_dataloader_pipeline(num_workers_candidates=[10, 8], train_batch_size_candidates=[64, 32], warm_up=1, epochs=3,
                                                device=device, transform=train_transform)

## 6. Hyperparameters Research

In [ ]:
optimal_hparams = hyperparameters_research(
    data_root='data_resized',
    learning_rate_stage1_candidates=[1e-4, 1e-3, 1e-2],
    weight_decay_rate_stage1_candidates=[1e-3, 1e-2],
    train_dir='train',
    val_dir='test',
    num_classes=2,
    precision='32-true',
    logger=False,
    enable_model_summary=False,
    enable_progress_bar=True,
    enable_checkpointing=False,
    weight_path='resnet50_weights.pth',
    train_transform=train_transform,
    val_transform=val_transform
)

## 7. Final Training Phase - Using Optimal Data Loader Pipeline Settings and Hyperparameters Values for Training

In [ ]:
# Instantiate and train the dataset and model again with best hyperparameter values

dataloader_pipeline_optimal_settings = load_dataloader_optimal_settings()
optimal_hparams = load_optimal_hyperparameters()

dataset, model, trainer = instantiate_and_train_dataset_and_model(
    data_root='data_resized',
    train_batch_size=dataloader_pipeline_optimal_settings['train_batch_size'],
    val_batch_size=dataloader_pipeline_optimal_settings['train_batch_size'], # In this project, val batch size=train batch size
    learning_rate_stage1=optimal_hparams['learning_rate_stage1'],
    learning_rate_stage2=optimal_hparams['learning_rate_stage2'], # In this project, learning_rate_stage2 = learning_rate_stage1 * 0.1
    num_epochs=10,
    train_transform=train_transform,
    val_transform=val_transform,
    devices=1,
    num_workers=dataloader_pipeline_optimal_settings['num_workers'],
    persistent_workers=dataloader_pipeline_optimal_settings['persistent_workers'],
    pin_memory=dataloader_pipeline_optimal_settings['pin_memory'],
    prefetch_factor=2,
    precision='32-true',
    weight_decay_rate_stage1=optimal_hparams['weight_decay_rate_stage1'],
    weight_decay_rate_stage2=optimal_hparams['weight_decay_rate_stage2'], # In this project, weight_decay_rate_stage2=weight_decay_rate_stage1
    logger=False,
    enable_model_summary=False,
    enable_progress_bar=True,
    enable_checkpointing=False,
    accumulate_grad_batches=2,
    train_dir='train', 
    val_dir='test',
    num_classes=2,
    weight_path='resnet50_weights.pth',
    fast_dev_run=False,
    save_weights_path='trained_model/custom_resnet50_trained_weights.pth'
)

## 8. Final Metrics and Sample Predictions

In [ ]:
# Final model training and validation accuracy (This cell only runs if you trained the model from scratch. If pretrained weights are loaded,
# trainer object is not returned by the function and therefore running this cell returns errors).

print(f'Final Epoch Train Accuracy: {trainer.callback_metrics['train_acc']:.2%}')
print(f'Final Epoch Validation Accuracy: {trainer.callback_metrics['val_acc']:.2%}')

In [ ]:
# Per-class classification accuracy on the validation set and confusion matrix visualization (This cell only runs if you trained the model from scratch.
# If pretrained weights are loaded, metric values are not returned and therefore running this cell returns errors).

cm = model.final_cm
per_class_acc = np.diag(cm) / np.sum(cm, axis=1)
print(f"Class 'fake' Prediction Accuracy: {per_class_acc[0]:.2%}")
print(f"Class 'real' Prediction Accuracy: {per_class_acc[1]:.2%}")

cm_display = ConfusionMatrixDisplay(cm, display_labels=np.array(['Fake', 'Real']))
cm_display.plot(cmap=plt.cm.Blues)
plt.show()
plt.close('all')

In [ ]:
# Predict and Draw Sample Images

sample_images_labels_predictions_visualization(trained_model=model, data_root='data', split='test', transform=val_transform, device=device)

In [ ]:
# Predict and visualize a sample image.

_ = show_and_predict_custom_img(img_path='inference/real/real3.jpg', trained_model=model, transform=val_transform, label='real', device=device)

## 9. Conclusion

In this notebook, we used [Tristan Zhang's AI generated images vs. real images kaggle dataset](https://www.kaggle.com/datasets/tristanzhang32/ai-generated-images-vs-real-images) for our classification project. We used `ResNet-50` pretrained as our baseline and used **2-stage fine-tuning** to fine-tune the model for the task at hand. Later, we searched the data loader pipeline settings and hyperparameters space and finally, with the optimal data loader pipeline settings and hyperparameters values found, we trained the final model, reported the metrics and showed sample predictions.